In [ ]:
!pip install lightgbm

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ─────────────────────────────────────────
# 1. Generate Synthetic Data (Loan Default)
# ─────────────────────────────────────────
X, y = make_classification(
    n_samples=10000,     # 10,000 customers
    n_features=15,       # 15 input features
    n_informative=10,    # 10 features are actually useful
    n_redundant=3,       # 3 redundant (noise) features
    random_state=42
)

# Assign meaningful feature names
feature_names = [
    'age', 'income', 'credit_score', 'loan_amount',
    'employment_years', 'num_accounts', 'debt_ratio',
    'payment_history', 'num_late_payments', 'credit_util',
    'num_inquiries', 'account_age', 'savings_balance',
    'monthly_expenses', 'collateral_value'
]

# Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['default'] = y  # target column: 1 = defaulted, 0 = not defaulted

print("Dataset shape  :", df.shape)
print("Default rate   :", df['default'].mean().round(3))

Dataset shape  : (10000, 16)
Default rate   : 0.5


In [3]:
df.head()

,age,income,credit_score,loan_amount,employment_years,num_accounts,debt_ratio,payment_history,num_late_payments,credit_util,num_inquiries,account_age,savings_balance,monthly_expenses,collateral_value,default
0,1.046080,1.599332,2.532688,-1.976311,0.687601,1.412669,1.522230,-0.899900,1.261921,-3.192460,0.344328,0.798130,-1.354848,2.396030,1.901351,1
1,0.359778,-0.861577,1.638884,0.445764,2.144193,-0.437632,-0.388196,0.724696,0.069161,0.874682,-0.888568,0.485751,0.327214,2.469600,-3.933467,0
2,1.516452,-4.385902,-0.535998,-1.017093,0.496900,4.593960,4.847821,2.065728,-0.840693,-0.354530,1.020490,-0.188313,-2.981050,3.154648,-1.948623,1
3,-0.269378,1.784872,-3.738618,0.437498,-7.610879,8.266258,1.485765,-0.830049,-0.145108,0.423583,-0.040116,4.617513,-2.550867,1.955587,2.813492,1
4,0.310363,-1.283132,0.903829,0.551854,0.666400,2.091127,2.909605,2.686448,1.191811,-2.558112,-1.050794,1.467806,-2.298239,4.178483,-1.573498,0


In [4]:
df.shape

(10000, 16)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df[feature_names], df['default'],
    test_size=0.2,           # 80% train, 20% test
    random_state=42,
    stratify=df['default']   # keep class distribution equal in both splits
)

print(f"\nTraining samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")


Training samples : 8000
Testing samples  : 2000


In [7]:
base_model = lgb.LGBMClassifier(
    n_estimators=200,    # build 200 trees
    random_state=42,
    verbose=-1           # suppress training logs
)

In [8]:
base_model.fit(X_train, y_train)

LGBMClassifier(n_estimators=200, random_state=42, verbose=-1)

In [9]:
base_preds = base_model.predict(X_test)
base_proba = base_model.predict_proba(X_test)[:, 1]  # probability of default

print("\n--- Base Model Performance ---")
print(f"ROC-AUC : {roc_auc_score(y_test, base_proba):.4f}")
print(classification_report(y_test, base_preds))


--- Base Model Performance ---
ROC-AUC : 0.9815
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1000
           1       0.94      0.94      0.94      1000

    accuracy                           0.94      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.94      0.94      0.94      2000



In [10]:
param_dist = {
    # Number of boosting trees to build
    'n_estimators': [100, 200, 300, 500],

    # Maximum tree depth (-1 means no limit, leaf-wise controls complexity)
    'max_depth': [-1, 5, 7, 10, 15],

    # Maximum number of leaves per tree (key LightGBM parameter)
    'num_leaves': [20, 31, 50, 70, 100],

    # Learning rate — how much each tree contributes to final prediction
    'learning_rate': [0.01, 0.05, 0.1, 0.2],

    # Fraction of features used per tree (controls overfitting)
    'feature_fraction': [0.6, 0.7, 0.8, 0.9, 1.0],

    # Fraction of training data used per tree (bagging)
    'bagging_fraction': [0.6, 0.7, 0.8, 0.9, 1.0],

    # Frequency of bagging (0 = disabled)
    'bagging_freq': [0, 3, 5, 7],

    # L1 regularization — encourages sparse feature weights
    'reg_alpha': [0, 0.01, 0.1, 0.5, 1.0],

    # L2 regularization — prevents large weights, reduces overfitting
    'reg_lambda': [0, 0.01, 0.1, 0.5, 1.0],

    # Minimum number of data points required in a leaf node
    'min_child_samples': [10, 20, 30, 50],
}


In [11]:
# Base estimator for tuning
lgbm_estimator = lgb.LGBMClassifier(
    random_state=42,
    verbose=-1
)


In [12]:
random_search = RandomizedSearchCV(
    estimator=lgbm_estimator,
    param_distributions=param_dist,
    n_iter=30,           # try 30 random combinations (not all)
    cv=5,                # 5-fold cross validation
    scoring='roc_auc',   # optimize for ROC-AUC score
    n_jobs=-1,           # use all available CPU cores
    random_state=42,
    verbose=1            # show progress
)

In [13]:
print("\nRunning RandomizedSearchCV... (this may take a minute)")
random_search.fit(X_train, y_train)

# Display best results
print(f"\nBest Cross-Validated ROC-AUC : {random_search.best_score_:.4f}")
print("\nBest Hyperparameters Found:")
for param, value in random_search.best_params_.items():
    print(f"  {param:25s}: {value}")


Running RandomizedSearchCV... (this may take a minute)
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best Cross-Validated ROC-AUC : 0.9826

Best Hyperparameters Found:
  reg_lambda               : 0
  reg_alpha                : 0.01
  num_leaves               : 70
  n_estimators             : 500
  min_child_samples        : 10
  max_depth                : -1
  learning_rate            : 0.05
  feature_fraction         : 1.0
  bagging_freq             : 5
  bagging_fraction         : 0.7


In [14]:
best_model = random_search.best_estimator_

# Predictions using the tuned model
tuned_preds = best_model.predict(X_test)
tuned_proba = best_model.predict_proba(X_test)[:, 1]

print("\n--- Tuned Model Performance ---")
print(f"ROC-AUC : {roc_auc_score(y_test, tuned_proba):.4f}")
print(classification_report(y_test, tuned_preds))


--- Tuned Model Performance ---
ROC-AUC : 0.9832
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1000
           1       0.94      0.94      0.94      1000

    accuracy                           0.94      2000
   macro avg       0.94      0.94      0.94      2000
weighted avg       0.94      0.94      0.94      2000

